# Bước 2 & 3: Huấn luyện (Training), Đánh giá và So sánh Mô hình
Notebook này đi thẳng vào trọng tâm: Nạp dữ liệu đã xử lý và chạy huấn luyện các thuật toán Học máy (Machine Learning) truyền thống bao gồm: **KNN, SVM, Random Forest** và kỹ thuật kết hợp bậc cao **Ensemble Learning**.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Thêm đường dẫn project để import code từ thư mục src
sys.path.append(os.path.abspath('..'))

# Import hàm tải dữ liệu trực tiếp từ file mã nguồn chính
from src.scripts.train_ensemble_pure import load_data
from src.core.config import CLASSES

# Các thư viện Machine Learning cốt lõi
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

## 1. Nạp Dữ liệu (Load Data)
Chúng ta sẽ gọi hàm `load_data()` để tự động quét toàn bộ thư mục, trích xuất đặc trưng (Màu sắc + Cấu trúc) và trả về bộ biến $X$ (đặc trưng) và $y$ (nhãn).

In [4]:
print("Đang nạp tập Huấn luyện (Train)...")
X_train, y_train = load_data('../dataset/train')

print("\nĐang nạp tập Đánh giá (Validation)...")
X_test, y_test = load_data('../dataset/validation')

# Chuẩn hóa dữ liệu (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nĐã nạp xong! Số lượng mẫu Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

Đang nạp tập Huấn luyện (Train)...

📂 Tải từ: ../dataset/train (Ensemble V1 Đa luồng)


  trash          : 100%|████████████████████| 478/478 [00:03<00:00, 122.69img/s]


  → 4376 samples

Đang nạp tập Đánh giá (Validation)...

📂 Tải từ: ../dataset/validation (Ensemble V1 Đa luồng)


  trash          : 100%|███████████████████████| 59/59 [00:04<00:00, 12.46img/s]


  → 541 samples

Đã nạp xong! Số lượng mẫu Train: 4376, Test: 541


## 2. Khởi tạo các Mô hình (Model Initialization)
Định nghĩa các thuật toán độc lập và ghép chúng lại bằng Ensemble Learning.

In [6]:
# Khởi tạo các mô hình cơ sở
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'SVM': SVC(kernel='rbf', C=10.0, probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
}

estimators = [(name, model) for name, model in models.items()]

# Khởi tạo Mô hình Kết hợp (Ensemble)
models['Voting Ensemble'] = VotingClassifier(estimators=estimators, voting='soft')
models['Stacking Ensemble'] = StackingClassifier(
    estimators=estimators, 
    final_estimator=LogisticRegression(), 
    cv=3
)

print("Đã khởi tạo Kiến trúc Học máy và Ensemble Learning!")

Đã khởi tạo Kiến trúc Học máy và Ensemble Learning!


## 3. Huấn luyện (Training) & So sánh
Gọi hàm `.fit()` cho từng mô hình và lưu lại độ chính xác để lập Bảng so sánh.

In [8]:
results = []
trained_models = {}

for name, model in models.items():
    print(f"Đang huấn luyện mô hình: {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({'Mô hình': name, 'Độ chính xác (Accuracy)': f"{acc * 100:.2f}%"})
    trained_models[name] = model
    
df_results = pd.DataFrame(results)
print("\n" + "="*40)
print("BẢNG SO SÁNH ĐÁNH GIÁ CÁC MÔ HÌNH HỌC MÁY")
print("="*40)
display(df_results.sort_values(by='Độ chính xác (Accuracy)', ascending=False).reset_index(drop=True))

Đang huấn luyện mô hình: KNN...
Đang huấn luyện mô hình: SVM...
Đang huấn luyện mô hình: Random Forest...
Đang huấn luyện mô hình: Voting Ensemble...


KeyboardInterrupt: 

## 4. Ma Trận Nhầm Lẫn (Confusion Matrix)
Phân tích lỗi sai của mô hình tốt nhất.

In [ ]:
best_model_name = 'Stacking Ensemble'
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, annot_kws={"size": 12})
plt.title(f'Ma Trận Nhầm Lẫn (Confusion Matrix) - {best_model_name}', fontsize=14, fontweight='bold')
plt.xlabel('Máy tính Dự đoán (Predicted)')
plt.ylabel('Thực tế (True Label)')
plt.show()